
# 03. Routing Modes Comparison
**Goal:** Compare the trade-offs of different routing strategies (`accuracy`, `fast`, `cheap`, `balanced`) on the same workload.


In [4]:
%load_ext autoreload
%autoreload 2
import sys
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Add project root to path
current_dir = Path(os.getcwd())
project_root = current_dir.parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from artemis_final.load_balancer.public_api import (
    ArtemisLoadBalancer,
    ModelCapacityConfig,
    StatsRegistry,
    RouterOutput,
    SchedulingContext,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload



## 1. Workload Definition
To ensure a fair comparison, we define a static **workload** (list of requests) that every mode will process.
We define 3 models:
- **Pro**: High Accuracy (0.95), High Latency (600ms), High Cost ($0.01)
- **Flash**: Low Accuracy (0.80), Low Latency (100ms), Low Cost ($0.001)
- **Balanced**: Med Accuracy (0.90), Med Latency (300ms), Med Cost ($0.005)


In [5]:

def get_stats_registry():
    stats = StatsRegistry()
    task = "vqa"
    # (lat, acc, cost)
    specs = {
        "pro_model": (600.0, 0.95, 0.01),
        "flash_model": (100.0, 0.80, 0.001),
        "balanced_model": (300.0, 0.90, 0.005),
    }
    for m, (lat, acc, cost) in specs.items():
        stats.update_latency(task, m, lat)
        stats.update_accuracy(task, m, acc)
        stats.update_cost(task, m, cost)
    return stats

def generate_workload(n=100):
    workload = []
    # Mix of preferences
    models = ["pro_model", "flash_model", "balanced_model"]
    
    for i in range(n):
        # Randomly pick a preferred model
        pref = np.random.choice(models)
        
        # Simulate some router probs roughly centered on preference
        probs = {m: 0.1 for m in models}
        probs[pref] = 0.8
        
        router_out = RouterOutput(
            router_probs=probs,
            preferred_model=pref,
            max_prob=0.8,
            sample_id='sim_req', task_type='vqa',
        )
        workload.append(router_out)
        
    return workload

# Pre-generate workload
workload = generate_workload(200)
print(f"Generated workload of {len(workload)} requests.")


Generated workload of 200 requests.



## 2. Running Simulations
We iterate through each mode, process the workload, and record statistics.


In [ ]:

modes = ["router_only", "accuracy", "fast", "cheap", "balanced"]
results = []
model_usage_by_mode = defaultdict(dict)

stats_reg = get_stats_registry()

# Define specs locally to match the previous cell for base_latency
specs = {
    "pro_model": 600.0,
    "flash_model": 100.0,
    "balanced_model": 300.0,
}

# Config: High capacity so queues don't dominate completely (we want to see logic choices)
configs = {}
for m, lat in specs.items():
    configs[m] = ModelCapacityConfig(
        model_name=m,
        base_latency_ms=lat,
        max_qps_per_replica=100.0,
        min_replicas=1,
        max_replicas=1
    )

# SLA that might pinch the 'pro_model' slightly
latency_sla = {"vqa": 1000.0, "default": 1000.0} 

for mode in modes:
    print(f"Running mode: {mode}...")
    lb = ArtemisLoadBalancer(
        model_configs=configs,
        stats_registry=stats_reg,
        latency_sla_ms=latency_sla,
        scheduling_mode=mode,
        simulation_only=True
    )
    
    decisions = []
    t = 0.0
    for i, router_out in enumerate(workload):
        t += 0.05 # 50ms inter-arrival
        ctx = SchedulingContext(
            arrival_ts_ms=t * 1000,
            load_profile="balanced",
            metadata={
                "sample_id": f"req_{i}",
                "task_type": "vqa",
            },
        )
        
        d = lb.schedule(router_out, ctx)
        decisions.append(d)
        
    # Aggregate Metrics
    avg_lat = np.mean([d.total_latency_ms for d in decisions])
    avg_cost = np.mean([d.est_cost_usd for d in decisions])
    avg_acc = np.mean([d.est_accuracy for d in decisions])
    violation_rate = np.mean([1.0 if d.sla_violated else 0.0 for d in decisions])
    
    results.append({
        "Mode": mode,
        "Avg Latency (ms)": avg_lat,
        "Avg Cost ($)": avg_cost,
        "Avg Accuracy": avg_acc,
        "SLA Violation %": violation_rate * 100
    })
    
    # Track usage
    usage = pd.Series([d.chosen_model for d in decisions]).value_counts()
    for m in configs.keys():
        model_usage_by_mode[mode][m] = usage.get(m, 0)

df_results = pd.DataFrame(results)
display(df_results)


Running mode: router_only...


TypeError: SchedulingContext.__init__() got an unexpected keyword argument 'sample_id'


## 3. Comparative Visualizations


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.barplot(data=df_results, x="Mode", y="Avg Latency (ms)", ax=axes[0,0], palette="viridis")
axes[0,0].set_title("Average Latency")

sns.barplot(data=df_results, x="Mode", y="Avg Cost ($)", ax=axes[0,1], palette="magma")
axes[0,1].set_title("Average Cost")

sns.barplot(data=df_results, x="Mode", y="Avg Accuracy", ax=axes[1,0], palette="RdBu")
axes[1,0].set_ylim(0.7, 1.0) # Zoom in
axes[1,0].set_title("Average Accuracy")

sns.barplot(data=df_results, x="Mode", y="SLA Violation %", ax=axes[1,1], palette="Reds")
axes[1,1].set_title("SLA Violation Rate")

plt.tight_layout()
plt.show()



## 4. Model Selection Behavior
Who gets chosen in which mode?


In [ ]:

df_usage = pd.DataFrame(model_usage_by_mode).T
df_usage.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab10')
plt.title("Model Selection Share per Mode")
plt.ylabel("Request Count")
plt.xlabel("Mode")
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.show()
